# 02. Data Cleaning and Preprocessing pipeline

This notebook focuses on applying robust data transformations to the raw Electric Vehicle Adoption dataset. 
In a professional Data Visualization and Analytics (DVA) workflow, accurate and consistent data is paramount for building trustworthy insights and dashboards. Every step taken below is meticulously documented to explain exactly **why** the transformation is required.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# Set up project root to easily import scripts
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.etl_pipeline import normalize_columns


## 1. Data Ingestion
**Action**: Load the raw dataset without modifying the underlying `data/raw` file.
**Why**: 
- **Preservation of Source Truth**: Operating purely on loaded DataFrames rather than altering source `csv` files ensures reproducibility and protects against destructive data loss.
- **Traceability**: If the cleaning logic must be revisited later, we have the identical raw snapshot to start from.


In [3]:
RAW_PATH = PROJECT_ROOT / 'data/raw/Electric_Vehicle_Population_Data.csv'
df = pd.read_csv(RAW_PATH)

print(f"Initial Dataset Shape: {df.shape}")
df.head(3)


Initial Dataset Shape: (280833, 16)


,VIN (1-10),County,City,State,Postal Code,Model Year,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligibility,Electric Range,Legislative District,DOL Vehicle ID,Vehicle Location,Electric Utility,2020 Census Tract
0,1N4AZ0CP6D,King,Kirkland,WA,98034.0,2013,NISSAN,LEAF,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,75.0,1.0,154635729,POINT (-122.22901 47.72201),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),5.303302e+10
1,5YJ3E1EC8L,Kitsap,Bainbridge Island,WA,98110.0,2020,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,308.0,23.0,107518645,POINT (-122.521 47.62759),PUGET SOUND ENERGY INC,5.303509e+10
2,5YJ3E1EBXJ,King,Seattle,WA,98144.0,2018,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,215.0,37.0,474808813,POINT (-122.30866 47.57874),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),5.303301e+10


## 2. Column Standardization & Basic Cleaning
**Action**: Convert all column names to standard `snake_case`, trim whitespaces from string columns, and drop completely duplicated rows.
**Why**: 
- **Namespace Consistency**: Column names with spaces, varying capitalizations, and brackets (e.g., `VIN (1-10)`) often lead to syntax errors during querying (like with `.query()` in Pandas) or in down-stream SQL workflows. Standardizing translates variables into a universally readable schema.
- **Reducing Skew**: Erroneous exact duplicates artificially inflate counts in our visualization aggregates.
- **Whitespace Stripping**: Hidden trailing or leading spaces in categorical columns disrupt grouped aggregations (e.g., `'Seattle '` vs `'Seattle'`).


In [4]:
# 1. Normalize columns using our modular pipeline function
df = normalize_columns(df)

# Rename problematic column specifically if not captured fully
if 'vin_1_10_' in df.columns:
    df.rename(columns={'vin_1_10_': 'vin_1_10'}, inplace=True)

# 2. Drop absolute duplicates
initial_len = len(df)
df = df.drop_duplicates()
print(f"Dropped {initial_len - len(df)} duplicate records.")

# 3. Strip whitespace from all string columns
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype("string").str.strip()
    
# Let's inspect the updated target columns
print("Normalized Columns:", df.columns.tolist())


Dropped 0 duplicate records.
Normalized Columns: ['vin_1_10', 'county', 'city', 'state', 'postal_code', 'model_year', 'make', 'model', 'electric_vehicle_type', 'clean_alternative_fuel_vehicle_cafv_eligibility', 'electric_range', 'legislative_district', 'dol_vehicle_id', 'vehicle_location', 'electric_utility', '2020_census_tract']


## 3. Geospatial Data Extraction
**Action**: The column `vehicle_location` commonly contains WKT string values like `POINT (-122.229 47.722)`. We will extract these values into separate `longitude` and `latitude` numeric columns, then drop the original string.
**Why**:
- **Tool Compatibility**: Most BI Tools, particularly Tableau, require independent `latitude` and `longitude` variables to intrinsically map data points without relying on complex string-parsing calculations natively inside the tool. Pulling them out during ETL optimizes dashboard performance.


In [5]:
if 'vehicle_location' in df.columns:
    # Use Regex to extract float values inside parentheses
    point_regex = r'POINT \((?P<longitude>[-\.\d]+) (?P<latitude>[-\.\d]+)\)'
    extracted_coords = df['vehicle_location'].str.extract(point_regex)
    
    # Cast to float
    df['longitude'] = extracted_coords['longitude'].astype(float)
    df['latitude'] = extracted_coords['latitude'].astype(float)
    
    # Drop original String location because it's no longer needed
    df = df.drop(columns=['vehicle_location'])
    print("Successfully extracted longitude and latitude features.")
else:
    print("vehicle_location column not found, skipped extraction.")


Successfully extracted longitude and latitude features.


## 4. Handling Missing/Null Data
**Action**: Address fields with NaNs (Not a Number) utilizing targeted imputation and omission strategies depending on the variable type and severity.
**Why**:
- **Dashboard Integrity**: Visualization tools usually fail or miscategorize nulls as a physical "Null" bar or point on the map. 
- **Actionable Accuracy**: 
    - Text variables (like `county`, `city`) will be tagged as `"Unknown"` precisely so they don't break aggregations but are explicitly identified as missing.
    - Minimal geographic missing variables (like `latitude`/`longitude`) might be dropped if the percentage is negligible, since an EV adoption map intrinsically requires a location.


In [6]:
# Provide a snapshot of missing values
missing_counts = df.isnull().sum()
print("Missing values before strategy applied:\n", missing_counts[missing_counts > 0])

# Strategy for Categorical Data
categorical_fills = {
    'county': 'Unknown',
    'city': 'Unknown',
    'legislative_district': -1, # Using -1 to represent unknown district integers
    'electric_utility': 'Unknown',
    '2020_census_tract': -1  
}

for col, fill_val in categorical_fills.items():
    if col in df.columns:
        df[col] = df[col].fillna(fill_val)

# Strategy for un-mapped instances (where long/lat is empty)
# Since this project focuses on spatial visualization, dropping unmappable tiny fractions is logical
coords_missing = df['longitude'].isnull().sum()
print(f"\nDropping {coords_missing} records missing Geospatial coordinates.")
df = df.dropna(subset=['longitude', 'latitude'])

print("\nMissing values after resolution:", df.isnull().sum().sum())


Missing values before strategy applied:
 county                   12
city                     12
postal_code              12
electric_range           12
legislative_district    703
electric_utility         12
2020_census_tract        12
longitude                20
latitude                 20
dtype: int64

Dropping 20 records missing Geospatial coordinates.

Missing values after resolution: 12


## 5. Standardizing Categorical Anomalies and Type Casting
**Action**: Convert `postal_code` and `model_year` into appropriate types. Create a Boolean or Categorical flag for zeroes in `electric_range`. 
**Why**:
- **Type Correctness**: `postal_code` as a continuous float logic assumes "98000" and "98001" have a mathematical relationship. They should be categorical strings.
- **Isolating Zero-Ranges**: The WA DOL stopped extensively measuring EV total battery ranges for newer iterations. Treating `0` range as an actual mathematical capability heavily skews range performance stats. Documenting whether range is "Known" vs "Unmeasured" unblocks precise vehicle metric analytics.


In [7]:
# Cast Postal Code to String format removing decimals
if 'postal_code' in df.columns:
    # Note: filling remaining NA's with 0 first to permit Float -> Int -> Str cast
    df['postal_code'] = df['postal_code'].fillna(0).astype("Int64").astype(str)
    df['postal_code'] = df['postal_code'].replace('0', 'Unknown')

if 'model_year' in df.columns:
    df['model_year'] = df['model_year'].astype('Int64')

# Handling '0' values in electric_range
if 'electric_range' in df.columns:
    # A Boolean flag indicating if Range statistics are structurally missing
    df['is_range_tracked'] = df['electric_range'].apply(lambda x: False if x == 0 else True)
    
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 280813 entries, 0 to 280832
Data columns (total 18 columns):
 #   Column                                           Non-Null Count   Dtype  
---  ------                                           --------------   -----  
 0   vin_1_10                                         280813 non-null  string 
 1   county                                           280813 non-null  string 
 2   city                                             280813 non-null  string 
 3   state                                            280813 non-null  string 
 4   postal_code                                      280813 non-null  object 
 5   model_year                                       280813 non-null  Int64  
 6   make                                             280813 non-null  string 
 7   model                                            280813 non-null  string 
 8   electric_vehicle_type                            280813 non-null  string 
 9   clean_alternative_fu

## 6. Exporting the Standardized Result
**Action**: Persist the processed, cleaned dataset to output structure `data/processed/` using an index-free CSV file format.
**Why**:
- **Modular Pipeline Stage**: Separation of concerns. End products (dashboards, EDA) shouldn't wait for ETL logic to re-run on demand. Loading the pre-processed output acts as a fast and definitive unified source of truth for the remainder of the analysis.


In [8]:
PROCESSED_PATH = PROJECT_ROOT / 'data/processed/cleaned_dataset.csv'

# Ensure the sub-directory exists before writing
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

# Export without index column to prevent cascading unnamed columns
df.to_csv(PROCESSED_PATH, index=False)

print(f"\U00002705 ETL Cleaning Finalized.")
print(f"Dataset securely exported to: {PROCESSED_PATH}")
print(f"Final Dimensionality: {df.shape[0]} rows, {df.shape[1]} columns.")


✅ ETL Cleaning Finalized.
Dataset securely exported to: /Users/jayadeep/Desktop/Section-C_G-6_EV-Adoption-Analytics/data/processed/cleaned_dataset.csv
Final Dimensionality: 280813 rows, 18 columns.
